In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import time
import ast
import seaborn as sns

from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA

## Вспомогательные штуки

In [ ]:
model = SentenceTransformer("all-MiniLM-l6-v2")
type(model)

In [ ]:
def get_pca_embeddings(model: SentenceTransformer, series_train: pd.Series, n_comp: int, prefix: str) -> pd.DataFrame:
    texts = series_train.astype(str).fillna("").tolist()
    emb_train = model.encode(texts, batch_size=64, show_progress_bar=True)
    
    pca = PCA(n_components=n_comp)
    pca.fit(emb_train)

    def get_pca_transform(series=series_train):
        if(series is series_train):
            pca_features = pca.transform(emb_train)
            cols = [f"{prefix}_pca_{i}" for i in range(n_comp)]
            return pd.DataFrame(pca_features, columns=cols, index=series.index)

        texts = series.astype(str).fillna("").tolist()
        emb = model.encode(texts, batch_size=64, show_progress_bar=True)
        pca_features = pca.transform(emb)
        cols = [f"{prefix}_pca_{i}" for i in range(n_comp)]
        return pd.DataFrame(pca_features, columns=cols, index=series.index)
    
    return get_pca_transform


def preprocessing(df: pd.DataFrame) -> pd.DataFrame:
    #df = df.copy().reset_index(drop=True)

    to_drop = [
        "header_image", "support_url", "support_email", "website", 
        "notes", "movies", "screenshots", "packages", "about_the_game", 
        "background", "detailed_description", "windows", "mac", "linux", "achievements", "recommendations", "user_score",
        "score_rank", "discount", "tags", "reviews"
    ]
    df = df.drop(columns=[c for c in to_drop if c in df.columns])

    df["release_date"] = pd.to_datetime(df["release_date"], format='%b %d, %Y', errors='coerce')
    now = pd.Timestamp.now()
    df["release_year"] = df["release_date"].dt.year
    df["release_month"] = df["release_date"].dt.month
    df["days_since_release"] = (now - df["release_date"]).dt.days.fillna(0)
    df = df.drop(columns=["release_date"])

    for col in ["supported_languages", "full_audio_languages"]:
        if col in df.columns:
            df[col] = df[col].astype(str).apply(
                lambda x: len(x.split(',')) if x.strip() and x != '[]' else 0
            )

    def process_owners(x):
        if '-' in str(x):
            try:
                parts = str(x).split('-')
                return sum(map(int, parts)) / 2
            except:
                return 0
        return 0

    df["estimated_owners_avg"] = df["estimated_owners"].apply(process_owners)
    df = df.drop(columns=["estimated_owners"])

    df["required_age"] = (df["required_age"] >= 18).astype(int)

    total_votes = df["positive"] + df["negative"]
    df["percent_positive"] = (df["positive"] / total_votes).fillna(0)
    df["metacritic_url"] = df["metacritic_url"].notna().astype(int)

    return df.drop_duplicates()


In [ ]:
def full_preprocessing(df,top_genre=None, top_cat=None, top_dev=None, top_pub=None, transformer_name=None, transformer_desc=None, scaler=None):

    df = preprocessing(df)
    df = df[df["price"] <= 200].copy()
    df = df[df["dlc_count"] <= 100].copy()
    df = df[df["peak_ccu"] > 0].copy()

    X = df.drop(["peak_ccu"], axis=1)
    y = df["peak_ccu"]

    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=101)

    if transformer_desc == None and transformer_name == None:
        transformer_name = get_pca_embeddings(model, X_train["name"], 32, "name")
        transformer_desc = get_pca_embeddings(model, X_train["short_description"], 5, "desc")

    x_name_emd = transformer_name(X_train ["name"])
    x_desc_emd = transformer_desc(X_train["short_description"])
    X_train = pd.concat([X_train, x_name_emd, x_desc_emd], axis=1)

    x_name_emd_t = transformer_name(X_test["name"])
    x_desc_emd_t = transformer_desc(X_test["short_description"])
    X_test = pd.concat([X_test, x_name_emd_t, x_desc_emd_t], axis=1)

    X_train = X_train.drop(["name", "short_description"], axis=1)
    X_test = X_test.drop(["name", "short_description"], axis=1)

    if top_genre is None:
        X_train["genres"] = X_train["genres"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
        X_test["genres"] = X_test["genres"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
        top_genre = X_train['genres'].explode().value_counts().nlargest(5).index.to_list()
    
    for genre in top_genre:  
        X_train[f'genre_{genre}'] = X_train['genres'].apply(lambda x: 1 if genre in x else 0)

    for genre in top_genre:
        X_test[f"genre_{genre}"] = X_test["genres"].apply(lambda x: 1 if genre in x else 0)

    if top_cat is None:
        X_train["categories"] = X_train["categories"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
        X_test["categories"] = X_test["categories"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
        top_cat = X_train['categories'].explode().value_counts().nlargest(5).index.to_list()

    for cat in top_cat:  
        X_train[f'cat_{cat}'] = X_train['categories'].apply(lambda x: 1 if cat in x else 0)

    for cat in top_cat:
        X_test[f"cat_{cat}"] = X_test["categories"].apply(lambda x: 1 if cat in x else 0)

    if top_dev is None:
        X_train["developers"] = X_train["developers"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
        X_test["developers"] = X_test["developers"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
        top_dev = X_train['developers'].explode().value_counts().nlargest(10).index.to_list()

    for dev in top_dev:  
        X_train[f'dev_{dev}'] = X_train['developers'].apply(lambda x: 1 if dev in x else 0)

    for dev in top_dev:
        X_test[f"dev_{dev}"] = X_test["developers"].apply(lambda x: 1 if dev in x else 0)

    if top_pub is None:
        X_train["publishers"] = X_train["publishers"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
        X_test["publishers"] = X_test["publishers"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
        top_pub = X_train['publishers'].explode().value_counts().nlargest(10).index.to_list()

    for pub in top_pub:  
        X_train[f'pub_{pub}'] = X_train['publishers'].apply(lambda x: 1 if pub in x else 0)

    for pub in top_pub:
        X_test[f"pub_{pub}"] = X_test["publishers"].apply(lambda x: 1 if pub in x else 0)

    X_train = X_train.drop(["developers", "publishers", "genres",
                            "categories","metacritic_url", "average_playtime_forever", 
                            "average_playtime_2weeks", "release_year",
                            "positive","negative"], axis=1)
    X_test = X_test.drop(["developers", "publishers", "genres",
                        "categories", "metacritic_url", "average_playtime_forever",
                          "average_playtime_2weeks", "release_year",
                            "positive","negative"], axis=1)
    
    X_train["median_playtime_2weeks"] = np.log1p(X_train["median_playtime_2weeks"])
    X_train["median_playtime_forever"] = np.log1p(X_train["median_playtime_forever"])

    X_test["median_playtime_2weeks"] = np.log1p(X_test["median_playtime_2weeks"])
    X_test["median_playtime_forever"] = np.log1p(X_test["median_playtime_forever"])

    X_train["estimated_owners_avg"] = np.log1p(X_train["estimated_owners_avg"])
    X_test["estimated_owners_avg"] = np.log1p(X_test["estimated_owners_avg"])

    y_train = np.log1p(y_train)
    y_test = np.log1p(y_test)
    from sklearn.preprocessing import StandardScaler

    print(X_train.columns)
    
    scaler = StandardScaler()
    scaler.fit(X_train)
    X_train_ = scaler.transform(X_train)
    X_test_ = scaler.transform(X_test)

    return (X_train_, X_test_, y_train, y_test, transformer_name, transformer_desc, scaler, top_genre, top_cat, top_dev, top_pub)



def full_preprocessing_test(df,top_genre, top_cat, top_dev, top_pub, transformer_name, transformer_desc, scaler):
    df = preprocessing(df)
    df = df[df["price"] <= 200].copy()
    df = df[df["dlc_count"] <= 100].copy()
    df = df[df["peak_ccu"] > 0].copy()

    X = df.drop(["peak_ccu"], axis=1)
    y = df["peak_ccu"]

    x_name_emd = transformer_name(X["name"])
    x_desc_emd = transformer_desc(X["short_description"])
    X = pd.concat([X, x_name_emd, x_desc_emd], axis=1)

    X = X.drop(["name", "short_description"], axis=1)
    
    for string in ["genres", "categories", "developers", "publishers"]:
        X[string] = X[string].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)


    for genre in top_genre:  
        X[f'genre_{genre}'] = X['genres'].apply(lambda x: 1 if genre in x else 0)

    for cat in top_cat:  
        X[f'cat_{cat}'] = X['categories'].apply(lambda x: 1 if cat in x else 0)

    for dev in top_dev:  
        X[f'dev_{dev}'] = X['developers'].apply(lambda x: 1 if dev in x else 0)

    for pub in top_pub:  
        X[f'pub_{pub}'] = X['publishers'].apply(lambda x: 1 if pub in x else 0)

    X = X.drop(["developers", "publishers", "genres",
                            "categories","metacritic_url", "average_playtime_forever", 
                            "average_playtime_2weeks", "release_year",
                            "positive","negative"], axis=1)
    
    X["median_playtime_2weeks"] = np.log1p(X["median_playtime_2weeks"])
    X["median_playtime_forever"] = np.log1p(X["median_playtime_forever"])

    X["estimated_owners_avg"] = np.log1p(X["estimated_owners_avg"])
    y = np.log1p(y)

    print(X.columns)
    from sklearn.preprocessing import StandardScaler
    X = scaler.transform(X)

    return (X, y)


## Преобразуем и потом разделяем

In [ ]:
df = pd.read_csv("../Data/2_lab.csv", index_col=False)

In [ ]:
df = preprocessing(df)

transformer_name = get_pca_embeddings(model, df["name"], 32, "name")
transformer_desc = get_pca_embeddings(model, df["short_description"], 5, "desc")

df_name_emd = transformer_name(df["name"])
df_desc_emd = transformer_desc(df["short_description"])
df = pd.concat([df, df_name_emd, df_desc_emd], axis=1)


In [ ]:
df = df.drop(["name", "short_description"], axis=1)

In [ ]:
df.info()

### Сжатие через логаримирование


In [ ]:
df = df[df["price"] <= 200]
df["price"] = np.log1p(df["price"])
df["price"].describe()

In [ ]:
df["average_playtime_forever"] = np.log1p(df["average_playtime_forever"])
df["average_playtime_2weeks"] = np.log1p(df["average_playtime_2weeks"])
df["median_playtime_forever"] = np.log1p(df["median_playtime_forever"])
df["median_playtime_2weeks"] = np.log1p(df["median_playtime_2weeks"])

In [ ]:
df = df[df["dlc_count"] <= 100]

In [ ]:
df["positive"] = np.log1p(df["positive"])
df["negative"] = np.log1p(df["negative"])

In [ ]:
df["peak_ccu"] = np.log1p(df["peak_ccu"])
df["peak_ccu"].describe()

### Обработка оставшихся колонок(специальная утечка данных)

In [ ]:
df['genres'] = df['genres'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
top_five = df["genres"].explode().value_counts().nlargest(5).index.to_list()
for genre in top_five:
    df[f'genre_{genre}'] = df['genres'].apply(lambda x: 1 if genre in x else 0)

In [ ]:
df['categories'] = df['categories'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
top_5 = df['categories'].explode().value_counts().nlargest(5).index.to_list()
for cat in top_5:  
    df[f'cat_{cat}'] = df['categories'].apply(lambda x: 1 if cat in x else 0)

In [ ]:
df = df.drop(["genres", "categories"], axis=1)

In [ ]:
df["developers"] = df["developers"].str.replace("[", "").str.replace("]", "").str.replace(" ", "")
df["developers"] = df["developers"].replace('', np.nan)

df['developers'] = df['developers'].fillna('Unknown')
top_devs = df['developers'].value_counts().nlargest(10).index
df['developers_grouped'] = df['developers'].apply(lambda x: x if x in top_devs else 'Other')

dev_dummies = pd.get_dummies(df['developers_grouped'], prefix='dev')
df = pd.concat([df, dev_dummies], axis=1)
df = df.drop(["developers", "developers_grouped"], axis=1)

df = df.drop(["dev_Other", "dev_Unknown"], axis=1)

In [ ]:
df["publishers"] = df["publishers"].str.replace("[", "").str.replace("]", "").str.replace(" ", "")
df["publishers"] = df["publishers"].replace('', np.nan)
df['publishers'] = df['publishers'].fillna('Unknown')

top_devs = df['publishers'].value_counts().nlargest(10).index
df['publishers_grouped'] = df['publishers'].apply(lambda x: x if x in top_devs else 'Other')
df["publishers_grouped"].value_counts()

dev_dummies = pd.get_dummies(df['publishers_grouped'], prefix='pub')
df = pd.concat([df, dev_dummies], axis=1)
df = df.drop(["publishers", "publishers_grouped"], axis=1)

df = df.drop(["pub_Other", "pub_Unknown"], axis=1)

In [ ]:
df.columns

### Кореляционный анализ

In [ ]:
corr_matrix = df.corr()

plt.figure(figsize=(15, 10))

sns.heatmap(
    corr_matrix, 
    annot=False,       
    cmap='coolwarm',   
    center=0,          
    fmt='.2f',         
    linewidths=0.5  
)

plt.title('Корреляция признаков датасета Steam')
plt.show()

In [ ]:
corr_matrix["metacritic_url"].nlargest(10)
df = df.drop(["metacritic_url"], axis=1)

In [ ]:
corr_matrix["metacritic_score"].nlargest(10)

In [ ]:
corr_matrix["average_playtime_2weeks"].nlargest(10)

In [ ]:
corr_matrix["average_playtime_forever"].nlargest(10)
df = df.drop(["average_playtime_2weeks", "average_playtime_forever"], axis=1)


In [ ]:
corr_matrix["percent_positive"].nlargest(10)

In [ ]:
corr_matrix["days_since_release"].nsmallest(10)

In [ ]:
corr_matrix["release_year"].nsmallest(10)
df = df.drop(["release_year"], axis = 1)

In [ ]:
corr_matrix["pub_'EroticGamesClub'"].nlargest(10)
df = df.drop(["pub_'EroticGamesClub'"], axis=1)

In [ ]:
corr_matrix["dev_'ChoiceofGames'"].nlargest(10)
df = df.drop(["dev_'ChoiceofGames'"], axis=1)

In [ ]:
df.columns

### Обучение

#### C нулевым пиком

In [ ]:
X = df.drop("peak_ccu", axis=1)
y = df["peak_ccu"]

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=101)

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
from sklearn.linear_model import ElasticNet
base_elastic_model = ElasticNet()

In [ ]:
param_grid = {"alpha":[i/10000 for i in range(10)],
               "l1_ratio":[0.1, 0.5, 0.7, 0.95, 0.99, 1]}

In [ ]:
from sklearn.model_selection import  GridSearchCV
grid_model = GridSearchCV(base_elastic_model, param_grid=param_grid, scoring = "neg_mean_squared_error", cv = 5, verbose = 2)

In [ ]:
grid_model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
y_pred = grid_model.predict(X_test)

In [ ]:
MSE = mean_squared_error(y_pred, y_test)
MAE = mean_absolute_error(y_pred, y_test)
RMSE = np.sqrt(MSE)
R2 = r2_score(y_test, y_pred)

print(f"MAE:{MAE}, MSE:{MSE}, RMSE:{RMSE}, R^2:{R2}")

In [ ]:
residuals = y_test - y_pred
plt.hist(residuals, bins=50)
plt.title('Распределение ошибок')
plt.show()

In [ ]:
grid_model.best_params_

#### Без нулевого пика

In [ ]:
df_without_zero = df[df["peak_ccu"] > 0]
df_without_zero

In [ ]:
X = df_without_zero.drop(["peak_ccu"], axis=1)
y = df_without_zero["peak_ccu"]

In [ ]:
X_ = X.drop(["pub_'Hede'"], axis=1)
X.columns

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=101)

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import ElasticNet
base_elastic_model = ElasticNet()

In [ ]:
param_grid = {"alpha":[i/10000 for i in range(10)],
               "l1_ratio":[0.1, 0.5, 0.7, 0.95, 0.99, 1]}

In [ ]:
from sklearn.model_selection import  GridSearchCV
grid_model = GridSearchCV(base_elastic_model, param_grid=param_grid, scoring = "neg_mean_squared_error", cv = 5, verbose = 0)

In [ ]:
grid_model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
y_pred = grid_model.predict(X_test)

In [ ]:
MSE = mean_squared_error(y_pred, y_test)
MAE = mean_absolute_error(y_pred, y_test)
RMSE = np.sqrt(MSE)
R2 = r2_score(y_test, y_pred)

print(f"MAE:{MAE}, MSE:{MSE}, RMSE:{RMSE}, R^2:{R2}")

In [ ]:
grid_model.best_params_

In [ ]:
residuals = y_test - y_pred
plt.hist(residuals, bins=50)
plt.title('Распределение ошибок')
plt.show()

## Разделяем и потом преобразуем

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("../Data/2_lab.csv", index_col=False)

In [ ]:
df = preprocessing(df)
df = df[df["price"] <= 200]
df = df[df["dlc_count"] <= 100]

In [ ]:
X = df.drop(["peak_ccu"], axis=1)
y = df["peak_ccu"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=101)

In [ ]:
transformer_name = get_pca_embeddings(model, X_train["name"], 32, "name")
transformer_desc = get_pca_embeddings(model, X_train["short_description"], 5, "desc")

In [ ]:
x_name_emd = transformer_name(X_train ["name"])
x_desc_emd = transformer_desc(X_train["short_description"])
X_train = pd.concat([X_train, x_name_emd, x_desc_emd], axis=1)

x_name_emd_t = transformer_name(X_test["name"])
x_desc_emd_t = transformer_desc(X_test["short_description"])
X_test = pd.concat([X_test, x_name_emd_t, x_desc_emd_t], axis=1)

In [ ]:
X_train = X_train.drop(["name", "short_description"], axis=1)
X_test = X_test.drop(["name", "short_description"], axis=1)

In [ ]:
X_test.columns

### Обработка оставшихся данных (уже без утечки данных)

In [ ]:
X_train["genres"]

In [ ]:
X_train["genres"] = X_train["genres"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
X_test["genres"] = X_test["genres"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
top_five = X_train['genres'].explode().value_counts().nlargest(5).index.to_list()

In [ ]:
for genre in top_five:  
    X_train[f'genre_{genre}'] = X_train['genres'].apply(lambda x: 1 if genre in x else 0)

for genre in top_five:
    X_test[f"genre_{genre}"] = X_test["genres"].apply(lambda x: 1 if genre in x else 0)

In [ ]:
X_train["categories"] = X_train["categories"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
X_test["categories"] = X_test["categories"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
top_five = X_train['categories'].explode().value_counts().nlargest(5).index.to_list()

In [ ]:
for cat in top_five:  
    X_train[f'cat_{cat}'] = X_train['categories'].apply(lambda x: 1 if cat in x else 0)

for cat in top_five:
    X_test[f"cat_{cat}"] = X_test["categories"].apply(lambda x: 1 if cat in x else 0)

In [ ]:
X_test.columns

In [ ]:
X_train["developers"] = X_train["developers"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
X_test["developers"] = X_test["developers"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
top_ten = X_train['developers'].explode().value_counts().nlargest(10).index.to_list()

In [ ]:
for dev in top_ten:  
    X_train[f'dev_{dev}'] = X_train['developers'].apply(lambda x: 1 if dev in x else 0)

for dev in top_ten:
    X_test[f"dev_{dev}"] = X_test["developers"].apply(lambda x: 1 if dev in x else 0)

In [ ]:
X_train["publishers"] = X_train["publishers"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
X_test["publishers"] = X_test["publishers"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
top_ten = X_train['publishers'].explode().value_counts().nlargest(10).index.to_list()

In [ ]:
for pub in top_ten:  
    X_train[f'pub_{pub}'] = X_train['publishers'].apply(lambda x: 1 if pub in x else 0)

for pub in top_ten:
    X_test[f"pub_{pub}"] = X_test["publishers"].apply(lambda x: 1 if pub in x else 0)

In [ ]:
X_train = X_train.drop(["developers", "publishers", "genres", "categories"], axis=1)
X_test = X_test.drop(["developers", "publishers", "genres", "categories"], axis=1)

In [ ]:
X_train.columns == X_test.columns

### Кореляционный анализ

In [ ]:
corr_matrix = X_train.corr()

plt.figure(figsize=(15, 10))

sns.heatmap(
    corr_matrix, 
    annot=False,       
    cmap='coolwarm',   
    center=0,          
    fmt='.2f',         
    linewidths=0.5  
)

plt.title('Корреляция признаков датасета Steam')
plt.show()

In [ ]:
X_train = X_train.drop(["metacritic_url"], axis=1)
X_test = X_test.drop(["metacritic_url"], axis=1)


In [ ]:
corr_matrix["average_playtime_2weeks"].nlargest(5)

In [ ]:
corr_matrix["average_playtime_forever"].nlargest(5)

In [ ]:
X_train = X_train.drop(["average_playtime_forever", "average_playtime_2weeks"], axis=1)
X_test = X_test.drop(["average_playtime_forever", "average_playtime_2weeks"], axis=1)

In [ ]:
corr_matrix["positive"].nlargest(10)

In [ ]:
corr_matrix["days_since_release"].nsmallest(10)

In [ ]:
X_train = X_train.drop(["release_year"], axis=1)
X_test = X_test.drop(["release_year"], axis=1)

In [ ]:
len(X_train.columns)

In [ ]:
corr_matrix["pub_EroticGamesClub"].nlargest(10)

In [ ]:
X_train = X_train.drop(["pub_EroticGamesClub"], axis=1)
X_test = X_test.drop(["pub_EroticGamesClub"], axis=1)

In [ ]:
corr_matrix["median_playtime_forever"].nlargest(10)

In [ ]:
corr_matrix["release_month"].nlargest(10)

In [ ]:
X_train = X_train.drop(["positive","negative"], axis=1)
X_test = X_test.drop(["positive","negative"], axis=1)

In [ ]:
print(len(X_train.columns))
X_train.columns

#### Логармирование

In [ ]:
X_train["median_playtime_2weeks"] = np.log1p(X_train["median_playtime_2weeks"])
X_train["median_playtime_forever"] = np.log1p(X_train["median_playtime_forever"])

X_test["median_playtime_2weeks"] = np.log1p(X_test["median_playtime_2weeks"])
X_test["median_playtime_forever"] = np.log1p(X_test["median_playtime_forever"])

In [ ]:
X_train["estimated_owners_avg"] = np.log1p(X_train["estimated_owners_avg"])
X_test["estimated_owners_avg"] = np.log1p(X_test["estimated_owners_avg"])

y_train = np.log1p(y_train)
y_test = np.log1p(y_test)

### Обучение


#### C нулевым пиком

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train)
X_train_ = scaler.transform(X_train)
X_test_ = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import ElasticNet
base_elastic_model = ElasticNet()

param_grid = {"alpha":[i/10000 for i in range(10)],
               "l1_ratio":[0.1, 0.5, 0.7, 0.95, 0.99, 1]}

In [ ]:
from sklearn.model_selection import  GridSearchCV
grid_model = GridSearchCV(base_elastic_model, param_grid=param_grid, scoring = "neg_mean_squared_error", cv = 5, verbose = 0)

In [ ]:
grid_model.fit(X_train_, y_train)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
y_pred = grid_model.predict(X_test_)

MSE = mean_squared_error(y_pred, y_test)
MAE = mean_absolute_error(y_pred, y_test)
RMSE = np.sqrt(MSE)
R2 = r2_score(y_test, y_pred)

print(f"MAE:{MAE}, MSE:{MSE}, RMSE:{RMSE}, R^2:{R2}")

In [ ]:
residuals = y_test - y_pred
plt.hist(residuals, bins=50)
plt.title('Распределение ошибок')
plt.show()

#### Без нулевого пика

In [ ]:
mask_train = y_train > 0
X_train__ = X_train[mask_train]
y_train__ = y_train[mask_train]

mask_test = y_test > 0
X_test__ = X_test[mask_test]
y_test__ = y_test[mask_test]

In [ ]:
#from sklearn.preprocessing import StandardScaler

#scaler = StandardScaler()
#scaler.fit(X_train__)
#X_train__ = scaler.transform(X_train__)
#X_test__ = scaler.transform(X_test__)

In [ ]:
# начинать отсюда
from sklearn.linear_model import ElasticNet
base_elastic_model = ElasticNet()

param_grid = {"alpha":[i/10000 for i in range(10)],
               "l1_ratio":[0.1, 0.5, 0.7, 0.95, 0.99, 1]}

In [ ]:
from sklearn.model_selection import  GridSearchCV
grid_model = GridSearchCV(base_elastic_model, param_grid=param_grid, scoring = "neg_mean_squared_error", cv = 5, verbose = 0)

In [ ]:
grid_model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
y_pred = grid_model.predict(X_test)

MSE = mean_squared_error(y_pred, y_test)
MAE = mean_absolute_error(y_pred, y_test)
RMSE = np.sqrt(MSE)
R2 = r2_score(y_test, y_pred)

print(f"MAE:{MAE}, MSE:{MSE}, RMSE:{RMSE}, R^2:{R2}")

In [ ]:
residuals = y_test - y_pred
plt.hist(residuals, bins=50)
plt.title('Распределение ошибок')
plt.show()

In [ ]:
grid_model.best_estimator_

## Авто обработка

In [ ]:
df = pd.read_csv("../Data/2_lab.csv", index_col=False)

tuple_values = full_preprocessing(df)

In [ ]:
game_data = {
    'name': ['Cyber-Samurai 2077'],
    'release_date': ['2026-04-07'],
    'required_age': [18],
    'price': [59.99],
    'dlc_count': [2],
    'detailed_description': ['A futuristic action RPG set in a neon-drenched metropolis.'],
    'about_the_game': ['Master the blade in a world of high-tech and low life.'],
    'short_description': ['Futuristic katana combat and deep RPG systems.'],
    'reviews': ['Very Positive'],
    'header_image': ['https://cdn.steam.com/cat.jpg'],
    'website': ['https://cybersamurai.com'],
    'support_url': ['https://support.cybersamurai.com'],
    'support_email': ['help@samurai.com'],
    'windows': [True],
    'mac': [False],
    'linux': [True],
    'metacritic_score': [88],
    'metacritic_url': ['https://metacritic.com/game/cybersamurai'],
    'achievements': [50],
    'recommendations': [150000],
    'notes': [None],
    'supported_languages': ["['English', 'Russian', 'Japanese']"],
    'full_audio_languages': ["['English', 'Japanese']"],
    'packages': [1],
    'developers': ["['Neon Katana Studios']"], 
    'publishers': ["['MegaCorp Games']"],
    'categories': ["['Single-player', 'Full controller support', 'Steam Cloud']"],
    'genres': ["['Action', 'RPG']"],
    'screenshots': ['img1.jpg'],
    'movies': ['video.mp4'],
    'user_score': [0],
    'score_rank': [None],
    'positive': [145000],
    'negative': [5000],
    'estimated_owners': ['1000000 .. 2000000'],
    'average_playtime_forever': [2500],
    'average_playtime_2weeks': [120],
    'median_playtime_forever': [1800],
    'median_playtime_2weeks': [45],
    'discount': [0],
    'peak_ccu': [85000], 
    'tags': ["['Cyberpunk', 'Action RPG', 'Difficult']"]
}

# Создаем DataFrame
df_custom = pd.DataFrame(game_data)

In [ ]:
X, y = full_preprocessing_test(df_custom, *tuple_values[7:], *tuple_values[4:7])

In [ ]:
import joblib
save_data = (*tuple_values[:4], X, y)

In [ ]:
joblib.dump(save_data, '../Data/preprocessing_all_info.joblib')